In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

In [ ]:
import os
import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from google.colab import files

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

RD_SEED = 42

os.environ['PYTHONHASHSEED'] = str(RD_SEED)

random.seed(RD_SEED)
np.random.seed(RD_SEED)

uploaded = files.upload()

TRAIN_FILE = "train_001_v1.csv"
TEST_FILE = "test_001_v1.csv"

train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

test_datasets = {
    TEST_FILE : test_df
}


required_columns = {
    "transaction_id",
    "sequence_no",
    "module",
    "function_name",
    "table_name",
    "field_changed",
    "action",
    "user_role",
    "trigger_source",
    "sequence_token",
    "label",
    "flow_id"
}

def validate_columns(df, dataset_name):
  missing_columns = [
      colmn
      for colmn in required_columns
      if colmn not in df.columns
  ]

  if missing_columns:
    raise ValueError(
        f"{dataset_name} is missing columns: "
        f"{missing_columns}"
    )



validate_columns(train_df, TRAIN_FILE)
validate_columns(test_df, TEST_FILE)

In [ ]:
def clean_trace_data(df):

  df = df.copy()
  df = df.dropna(subset=["transaction_id"])

  df["transaction_id"] = (

    df["transaction_id"]
    .astype(str)
    .str.strip()
  )

  df["sequence_no"] = pd.to_numeric(
      df["sequence_no"],
      errors="coerce"
  )

  df["sequence_no"] = (
      df.groupby("transaction_id")["sequence_no"]
      .transform(
          lambda values:
          values.fillna(
              pd.Series(
                  range(1, len(values) + 1),
                  index=values.index,
                  dtype=float
              )
          )
      )
  )

  text_columns = [
      "module",
      "function_name",
      "table_name",
      "field_changed",
      "action",
      "user_role",
      "trigger_source",
      "sequence_token",
      "old_value",
      "new_value",
      "flow_id",
      "label"
  ]


  for col in text_columns:

    if col not in df.columns:
      df[col] = "UNKNOWN"

    df[col] = (
        df[col]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
    )


  df["action"] = df["action"].str.upper()
  df["label"]=df["label"].str.lower()
  df["flow_id"]=df["flow_id"].str.upper()

  df = df.sort_values(
        by=["transaction_id", "sequence_no"]
  ).reset_index(drop=True)

  return df

train_df = clean_trace_data(train_df)

for filename in test_datasets:
    test_datasets[filename] = clean_trace_data(
        test_datasets[filename]
)

In [ ]:
def count_action(values, keyword):

  values = values.astype(str).str.upper()

  return values.str.contains(
      keyword,
      regex=False,
      na=False
  ).sum()


def count_text_pattern(values, keyword):

  values = values.astype(str).str.lower()

  return values.str.contains(
      keyword.lower(),
      regex=False,
      na=False
  ).sum()


def consecutive_repeat_count(val):

  val = val.astype(str).tolist()

  if len(val) == 0:
    return 0

  return sum(
      val[index] ==  val[index - 1]
      for index in range(1 , len(val))
   )

def transition_count(values):

    values = values.astype(str).tolist()

    if len(values) <= 1:
      return 0

    return sum(
        values[index] != values[index -1]
        for index in range(1, len(values))
    )

def repeated_event_count(values):

    counts = values.astype(str).value_counts()

    return int(
        (counts - 1)
        .clip(lower=0)
        .sum()
    )

In [ ]:
def extract_transaction_features(df):

  feature_rows = []

  grouped_transactions = df.groupby(
      "transaction_id",
      sort=False
  )

  for transaction_id, group in grouped_transactions:

    group = group.sort_values("sequence_no")

    actions = group["action"]
    functions = group["function_name"]
    modules = group["module"]
    tables = group["table_name"]
    fields = group["field_changed"]
    roles = group["user_role"]
    triggers = group["trigger_source"]
    tokens = group["sequence_token"]

    sequence_length = len(group)
    unique_tokens = tokens.nunique()

    feature_row = {
        "transaction_id" : transaction_id,
        "sequence_length" : sequence_length,
        "unique_token_count": unique_tokens,
        "unique_function_count": functions.nunique(),
        "unique_module_count" : modules.nunique(),
        "unique_table_count" : tables.nunique(),
        "unique_field_count": fields.nunique(),
        "unique_action_count": actions.nunique(),
        "unique_role_count": roles.nunique(),
        "unique_trigger_count" : triggers.nunique(),

        "repeated_token_count" : repeated_event_count(tokens),
        "repeated_functions_count": repeated_event_count(functions),
        "consecutive_token_repeats": consecutive_repeat_count(tokens),
        "consecutive_function_repeats": consecutive_repeat_count(functions),
        "token_transition_count": transition_count(tokens),
        "function_transition_count": transition_count(functions),
        "module_transition_count": transition_count(modules),
        "table_transition_count": transition_count(tables),
        "role_transition_count": transition_count(roles),
        "create_count": count_action(actions, "CREATE"),
        "write_count": count_action(actions, "WRITE"),
        "update_count": count_action(actions, "UPDATE"),
        "read_count": count_action(actions, "READ"),
        "delete_count": count_action(actions, "DELETE"),
        "confirm_count": count_action(actions, "CONFIRM"),
        "validate_count": count_action(actions, "VALIDATE"),
        "post_count": count_action(actions, "POST"),
        "state_change_count": count_action(actions, "STATE"),
        "purchase_module_count": count_text_pattern(modules, "purchase"),
        "stock_module_count": count_text_pattern(modules, "stock"),
        "account_module_count": count_text_pattern(modules, "account"),
        "purchase_order_count": count_text_pattern(
            tables,
            "purchase.order"
        ),
        "purchase_line_count": count_text_pattern(
            tables,
            "purchase.order.line"
        ),

        "stock_picking_count": count_text_pattern(
            tables,
            "stock.picking"
        ),
        "stock_move_count": count_text_pattern(
            tables,
            "stock.move"
        ),
        "account_move_count": count_text_pattern(
            tables,
            "account.move"
        ),
        "quantity_change_count": count_text_pattern(
            fields,
            "product_qty"
        ),
        "price_change_count" : count_text_pattern(
            fields,
            "price_unit"
        ),
        "planned_date_change_count": count_text_pattern(
                fields,
                "date_planned"
        ),

        "vendor_change_count" : (
            count_text_pattern(
                fields,
                "partner_id"
            )
            +
            count_text_pattern(
                fields,
                "vendor"
            )
        ),
        "extreme_value_count": count_text_pattern(
            group["new_value"],
            "extreme"
        ),
        "high_value_count": count_text_pattern(
            group["new_value"],
            "high"
        ),
        "state_value_count": count_text_pattern(
            fields,
            "state"
        ),

    }

    denominator = max(sequence_length, 1)

    feature_row["unique_token_ratio"] = (
        unique_tokens / denominator
    )

    feature_row["repeated_token_ratio"] = (
        feature_row["repeated_token_count"] / denominator
    )

    feature_row["write_ratio"] = (
        feature_row["write_count"]
        / denominator
    )

    feature_row["create_ratio"] = (
        feature_row["create_count"] / denominator
    )


    feature_row["purchase_ratio"] = (
        feature_row["purchase_module_count"] / denominator
    )

    feature_row["stock_ratio"] = (
        feature_row["stock_module_count"] / denominator
    )

    feature_row["account_ratio"] = (
        feature_row["account_module_count"] / denominator
    )

    feature_rows.append(feature_row)

  features = pd.DataFrame(feature_rows)

  return features



train_features = extract_transaction_features(train_df)

print("\n Training transaction features:")

print("Transactions:", len(train_features))

print("Features:", len(train_features.columns) - 1)

display(train_features.head())


train_labels = (
    train_df.groupby("transaction_id")["label"]
    .first()
    .reset_index()
)

train_features = train_features.merge(
    train_labels,
    on="transaction_id",
    how="left"
)

normal_train_features = train_features[
    train_features["label"] == "normal"
].copy()

if len(normal_train_features) == 0:
  raise ValueError(
      "No normal transactions were found in the training file"
  )

print("\nTraining label distribution:")
print(train_features["label"].value_counts())

print("\nNormal transactions used for Isolation Forest training:",
  len(normal_train_features))

non_feature_columns = [
    "transaction_id",
    "label",
]

feature_columns = [
    column
    for column in normal_train_features.columns
    if column not in non_feature_columns
]


In [ ]:
RD_SEED = 42
VALIDATION_FRACTION = 0.20
TARGET_VALIDATION_FPR = 0.05

train_part, validation_part = train_test_split(
    normal_train_features,
    test_size=VALIDATION_FRACTION,
    random_state=RD_SEED,
    shuffle=True
)

X_train = train_part[feature_columns].replace([np.inf, -np.inf], np.nan).fillna(0)
X_val = validation_part[feature_columns].replace([np.inf, -np.inf], np.nan).fillna(0)

print("\nModel feature columns:")
print(feature_columns)
print("Normal model-fitting transactions:", len(X_train))
print("Normal threshold-calibration transactions:", len(X_val))
assert set(train_part["transaction_id"]).isdisjoint(set(validation_part["transaction_id"]))

In [ ]:
isolation_forest = IsolationForest(
    n_estimators=500,
    max_samples="auto",
    contamination="auto",
    max_features=1.0,
    bootstrap=False,
    random_state=RD_SEED,
    n_jobs=-1,
)
isolation_forest.fit(X_train)
print("\nIsolation Forest training completed.")

In [ ]:
train_anomaly_scores = -isolation_forest.score_samples(X_train)
validation_anomaly_scores = -isolation_forest.score_samples(X_val)

threshold_percentile = 100 * (1 - TARGET_VALIDATION_FPR)
anomaly_threshold = float(np.percentile(validation_anomaly_scores, threshold_percentile))

In [ ]:
validation_false_positive_rate = float(np.mean(validation_anomaly_scores >= anomaly_threshold))
print("\nThreshold calibration (NORMAL VALIDATION ONLY):")
print("Target validation false-positive rate:", TARGET_VALIDATION_FPR)
print("Percentile:", threshold_percentile)
print("Selected anomaly threshold:", round(anomaly_threshold, 6))
print("Observed validation false-positive rate:", round(validation_false_positive_rate, 4))

training_predictions = np.where(
    train_anomaly_scores >= anomaly_threshold, "anomaly", "normal"
)
training_false_positive_rate = float(np.mean(training_predictions == "anomaly"))
print("Observed training false-positive rate:", round(training_false_positive_rate, 4))

In [ ]:
def predict_dataset(
    raw_df,
    dataset_name,
    trained_model,
    threshold,
    model_feature_columns
):

  transaction_features = extract_transaction_features(
      raw_df
  )

  transaction_labels = (
      raw_df.groupby("transaction_id")["label"]
      .first()
      .reset_index()
  )

  transaction_flows = (
      raw_df.groupby("transaction_id")["flow_id"]
      .first()
      .reset_index()
  )


  results = transaction_features.merge(
      transaction_labels,
      on="transaction_id",
      how="left"
  )

  results = results.merge(
      transaction_flows,
      on="transaction_id",
      how="left"
  )


  X_test = results[model_feature_columns].copy()

  X_test = X_test.replace(
      [np.inf, -np.inf],
      np.nan
  )

  X_test = X_test.fillna(0)


  normality_scores = trained_model.score_samples(
      X_test
  )

  anomaly_scores = -normality_scores

  predicted_labels = np.where(
      anomaly_scores >= threshold,
      "anomaly",
      "normal"
  )

  results["normality_score"] = normality_scores
  results["anomaly_score"] = anomaly_scores
  results["anomaly_threshold"] = threshold
  results["predicted_label"] = predicted_labels

  results["prediction_correct"] = (
      results["label"] == results["predicted_label"]
  )

  results["dataset"] = dataset_name

  return results

In [ ]:
def evaluate_results(results, dataset_name):

  print("\n" + "=" * 70)
  print("Results for:",  dataset_name)
  print("=" * 70)


  y_true = results["label"]
  y_pred = results["predicted_label"]

  valid_labels = { "normal" , "anomaly"}

  valid_mask = (
      y_true.isin(valid_labels) & y_pred.isin(valid_labels)
  )

  evaluation_results = results[ valid_mask ].copy()

  if len(evaluation_results) == 0:
    print("No valid normal/anomaly labels were found.")
    return None


  y_true = evaluation_results["label"]
  y_pred = evaluation_results["predicted_label"]


  accuracy = accuracy_score(
      y_true,
      y_pred
  )

  precision = precision_score(
      y_true,
      y_pred,
      pos_label="anomaly",
      zero_division=0
  )

  recall = recall_score(
      y_true,
      y_pred,
      pos_label="anomaly",
      zero_division=0
  )

  f1 = f1_score(
      y_true,
      y_pred,
      pos_label="anomaly",
      zero_division=0
  )

  if y_true.nunique() == 2:
    y_binary = (y_true == "anomaly").astype(int)
    roc_auc = roc_auc_score(y_binary, evaluation_results["anomaly_score"])
    pr_auc = average_precision_score(y_binary, evaluation_results["anomaly_score"])
  else:
    roc_auc = np.nan
    pr_auc = np.nan

  print("\n Transaction counts:")
  print("Total transactions:", len(evaluation_results))
  print("\nActual labels:")
  print(y_true.value_counts())


  print("\n Predicted labels:")
  print(y_pred.value_counts())

  print("\n Overall metrics:")
  print(f"Accuracy : {accuracy:.4f} ")
  print(f"Precision : {precision:.4f} ")
  print(f"Recall : {recall:.4f} ")
  print(f"F1-score : {f1:.4f} ")
  print(f"ROC-AUC : {roc_auc:.4f} ")
  print(f"Average precision (PR-AUC summary) : {pr_auc:.4f} ")

  print("\n Classification report:")

  print(
      classification_report(
          y_true,
          y_pred,
          labels=["normal", "anomaly"],
          zero_division=0
      )
  )

  matrix = confusion_matrix(
      y_true,
      y_pred,
      labels=["normal" , "anomaly"]
  )

  display_object = ConfusionMatrixDisplay(

    confusion_matrix=matrix,
    display_labels=["Normal", "Anomaly"]
  )

  print("Confusion matrix:")
  print(matrix)

  display_object.plot(
      values_format = "d"
  )

  plt.title(
      f"Isolation Forest confusion matrix \n {dataset_name}"
  )

  plt.tight_layout()
  plt.savefig(
    "pic1.png",
    dpi=600,
    bbox_inches="tight"
  )

  plt.show()

  metric_summary = pd.DataFrame({
      "dataset" : [dataset_name],
      "accuracy": [accuracy],
      "precision" : [precision],
      "recall" : [recall],
      "f1_score" : [f1],
      "roc_auc": [roc_auc],
      "average_precision": [pr_auc],
      "total_transactions" : [
          len(evaluation_results)
      ],
      "actual_normal" : [
          (y_true == "normal").sum()
      ],
      "actual_anomaly" : [
          (y_true == "anomaly").sum()
      ],
      "predicted_normal": [
          (y_pred == "normal").sum()
      ],
      "predicted_anomaly": [
          (y_pred == "anomaly").sum()
      ]
  })

  return metric_summary


In [ ]:
def create_scenario_summary(results):

  summary_rows = []

  available_flows = (
      results["flow_id"]
      .replace("UNKNOWN", np.nan)
      .dropna()
      .unique()
  )

  for flow in sorted(available_flows):

    subset = results[
        results["flow_id"] == flow
    ].copy()


    total_transactions = len(subset)


    actual_anomalies = (
        subset["label"] == "anomaly"
    ).sum()


    actual_normals = (
        subset["label"] == "normal"
    ).sum()

    predicted_anomalies = (
        subset["predicted_label"] == "anomaly"
    ).sum()

    correctly_detected_anomalies = (
        (
            subset["label"] == "anomaly"
        ) & (
            subset["predicted_label"] == "anomaly"
        )
    ).sum()

    missed_anomalies = (
        (subset["label"] == "anomaly") & (subset["predicted_label"] == "normal")
    ).sum()

    false_positives = (
        (subset["label"] == "normal") & ( subset["predicted_label"] == "anomaly")
    ).sum()


    correctly_classified = (
        subset["label"] == subset["predicted_label"]
    ).sum()


    if actual_anomalies > 0 :
      anomaly_detection_rate = (
          correctly_detected_anomalies / actual_anomalies
      )
    else:
      anomaly_detection_rate = np.nan



    if actual_normals > 0 :
      false_positive_rate = (
          false_positives / actual_normals
      )
    else:
      false_positive_rate = np.nan


    scenario_accuracy = (

      correctly_classified / total_transactions
      if total_transactions > 0
      else np.nan
    )


    summary_rows.append({
        "flow_id": flow,
        "total_transactions" : total_transactions,
        "actual_normal": actual_normals,
        "actual_anomaly" : actual_anomalies,
        "predicted_anomaly": predicted_anomalies,
        "correctly_detected_anomaly" : correctly_detected_anomalies,
        "missed_anomaly": missed_anomalies,
        "false_positives": false_positives,
        "detection_rate": anomaly_detection_rate,
        "false_positive_rate": false_positive_rate,
        "scenario_accuracy": scenario_accuracy
    })

  return pd.DataFrame(summary_rows)


all_results = []
all_metric_summaries = []
all_scenario_summaries = []


for filename, test_df in test_datasets.items():

  results = predict_dataset(
      raw_df = test_df,
      dataset_name=filename,
      trained_model=isolation_forest,
      threshold=anomaly_threshold,
      model_feature_columns=feature_columns
  )

  all_results.append(results)

  metric_summary = evaluate_results(
      results,
      filename
  )

  if metric_summary is not None:
    all_metric_summaries.append(
        metric_summary
    )

  scenario_summary = create_scenario_summary(results)

  scenario_summary.insert(
      0,
      "dataset",
      filename
  )

  all_scenario_summaries.append( scenario_summary)


  print("\nPer-scenario results:")
  display(scenario_summary)


combined_results = pd.concat(
    all_results,
    ignore_index=True
)

if len(all_metric_summaries) > 0:

  combined_metrics = pd.concat(
      all_metric_summaries,
      ignore_index=True
  )

else:
  combined_metrics = pd.DataFrame()

if len(all_scenario_summaries) >0:
  combined_scenario_summary = pd.concat(
      all_scenario_summaries,
      ignore_index=True
  )
else:
  combined_scenario_summary = pd.DataFrame()


print("\n" + "=" * 70)
print("comparison between test datasets")
print("=" * 70)

display(combined_metrics)

print("\n" + "=" * 70)
print("combined per scenario results")
print("=" * 70)

display(combined_scenario_summary)

anomaly_scenarios = [
    "A1",
    "A2",
    "A3",
    "A4",
    "A5"
]

a1_to_a5_summary = combined_scenario_summary[
    combined_scenario_summary["flow_id"].isin(
        anomaly_scenarios
    )
].copy()

if len(a1_to_a5_summary) > 0:

  print("\n A1-A5 comparison:")

  display(
      a1_to_a5_summary[
          [
              "dataset",
              "flow_id",
              "actual_anomaly",
              "correctly_detected_anomaly",
              "missed_anomaly",
              "detection_rate"
          ]
      ]
  )

  detection_pivot = a1_to_a5_summary.pivot(
      index="flow_id",
      columns="dataset",
      values="detection_rate"
  )

  print("\n A1-A5 detection rate pivot table:")
  display(detection_pivot)

else:
    print("\n A1-A5 summary was not created because the flow_id column does not contain A1-A5.")


for filename in combined_results["dataset"].unique():

    subset = combined_results[
        combined_results["dataset"] == filename
    ]

    normal_scores = subset.loc[
        subset["label"] == "normal",
        "anomaly_score"
    ]

    anomaly_scores = subset.loc[
        subset["label"] == "anomaly",
        "anomaly_score"
    ]

    plt.figure(figsize=(9,5))

    if len(normal_scores) > 0:

      plt.hist(
          normal_scores,
          bins=30,
          alpha=0.6,
          label="Normal"
      )

    if len(anomaly_scores) > 0:

        plt.hist(
            anomaly_scores,
            bins=30,
            alpha=0.6,
            label="Anomaly"
        )

    plt.axvline(
            anomaly_threshold,
            linestyle="--",
            label="Threshold"
        )

    plt.xlabel("Anomaly Score")
    plt.ylabel("Number of Transactions")

    plt.title(
            f"Isolation Forest Anomaly Scores\n{filename}"
        )

    plt.legend()
    plt.tight_layout()
    plt.savefig(
     "pic2.png",
      dpi=600,
      bbox_inches="tight"
    )
    plt.show()

feature_explanation = combined_results[
          feature_columns + ["anomaly_score"]
      ].copy()

feature_correlations = (
          feature_explanation
          .corr(numeric_only=True)["anomaly_score"]
          .drop("anomaly_score")
          .abs()
          .sort_values(ascending=False)
          .reset_index()
      )

feature_correlations.columns = [
          "feature",
          "absolute_score_correlation"
      ]
print("\nFeatures most associated with anomaly scores:")
display(feature_correlations.head(15))

most_anomalous = combined_results.sort_values(
          "anomaly_score",
          ascending=False
      )

columns_to_display = [
          "dataset",
          "transaction_id",
          "flow_id",
          "label",
          "predicted_label",
          "anomaly_score",
          "sequence_length",
          "unique_token_count",
          "repeated_token_count",
          "write_count",
          "vendor_change_count",
          "extreme_value_count"
      ]


print("\n Most anomalous transactions:")

display(
          most_anomalous[ columns_to_display].head(30)
      )


In [ ]:
#A4
a4 = results[
    results["flow_id"].astype(str).str.upper() == "A4"
].copy()


a4["outcome"] = a4["predicted_label"].map({
    "normal": "MISSED",
    "anomaly": "DETECTED"
})

print("\nA4 transaction-level results")

display(
    a4[
        [
            "transaction_id",
            "flow_id",
            "sequence_length",
            "anomaly_score",
            "predicted_label",
            "outcome"
        ]
    ].sort_values("anomaly_score")
)

print("\nA4 detection summary")

display(
    a4.groupby("outcome").agg(
        transactions=("transaction_id", "count"),
        mean_length=("sequence_length", "mean"),
        mean_score=("anomaly_score", "mean"),
        min_score=("anomaly_score", "min"),
        max_score=("anomaly_score", "max")
    )
)

feature_columns = [
    "sequence_length",
    "unique_token_count",
    "unique_function_count",
    "repeated_token_count",
    "repeated_function_count",
    "consecutive_repeat_count"
]

feature_columns = [
    col for col in feature_columns
    if col in a4.columns
]

print("\nA4 feature comparison")

display(
    a4.groupby("outcome")[feature_columns]
    .agg(["mean", "min", "max"])
    .T
)


In [ ]:
a4 = results.loc[
    results["flow_id"] == "A4"
].copy()

a4["outcome"] = a4["predicted_label"].map({
    "normal": "MISSED",
    "anomaly": "DETECTED"
})

a4_feature_cols = [
    c for c in feature_columns
    if c in a4.columns
]

print("Model features:", len(a4_feature_cols))

print(
    "Distinct A4 feature vectors:",
    a4[a4_feature_cols].drop_duplicates().shape[0]
)

comparison = (
    a4.groupby("outcome")[a4_feature_cols]
    .mean()
    .T
)

comparison["difference"] = (
    comparison["DETECTED"] - comparison["MISSED"]
)

print("\nFeatures that differ between outcomes:")

display(
    comparison[
        comparison["difference"].abs() > 1e-10
    ].sort_values("difference")
)

repeat_cols = [
    c for c in a4_feature_cols
    if "repeat" in c.lower()
]

print("\nRepetition-related features:")

if repeat_cols:
    display(
        a4.groupby("outcome")[repeat_cols]
        .agg(["mean", "min", "max"])
        .T
    )
else:
    print("No repetition-related features found.")


In [ ]:
#A4
compare_ids = ["P00242", "P00241"]

for tid in compare_ids:
    print("\n" + "=" * 85)
    print("Transaction:", tid)

    transaction = test_df[
        test_df["transaction_id"] == tid
    ].sort_values("sequence_no")

    for _, row in transaction.iterrows():
        print(
            f"{int(row['sequence_no']):02d} "
            f"{row['sequence_token']}"
        )


In [ ]:

#A4
df = combined_results.copy()

if DATASET_NAME is not None:
    df = df[df["dataset"] == DATASET_NAME].copy()


a4 = df[df["flow_id"].astype(str).str.upper() == "A4"].copy()

a4["outcome"] = np.where(
    a4["predicted_label"] == "anomaly",
    "DETECTED",
    "MISSED"
)

display(
    a4[[
        "transaction_id",
        "sequence_length",
        "anomaly_score",
        "predicted_label",
        "outcome"
    ]].sort_values("anomaly_score")
)

selected_features = [
    "sequence_length",
    "unique_token_count",
    "repeated_token_count",
    "repeated_functions_count",
    "consecutive_function_repeats",
    "consecutive_token_repeats",
    "planned_date_change_count",
    "write_count",
    "purchase_line_count",
    "purchase_module_count",
    "purchase_order_count",
    "token_transition_count"
]

selected_features = [
    f for f in selected_features if f in a4.columns
]

print("\n=== FEATURE MEANS BY OUTCOME ===")

comparison = (
    a4.groupby("outcome")[selected_features]
    .mean()
    .T
)

comparison["difference"] = (
    comparison.get("DETECTED", np.nan)
    - comparison.get("MISSED", np.nan)
)

display(comparison.round(4))

print("\n=== A4 FEATURE-SCORE CORRELATIONS ===")

correlations = (
    a4[selected_features + ["anomaly_score"]]
    .corr(method="spearman")["anomaly_score"]
    .drop("anomaly_score")
    .dropna()
    .sort_values(key=abs, ascending=False)
)

display(
    correlations.rename("spearman_correlation")
    .reset_index()
    .rename(columns={"index": "feature"})
)


display(
    a4[
        a4["transaction_id"].isin(
            ["P00241", "P00242"]
        )
    ][
        ["transaction_id", "outcome", "anomaly_score"]
        + selected_features
    ].set_index("transaction_id").T
)

if DATASET_NAME is not None:
    raw_test = test_datasets[DATASET_NAME]
elif len(test_datasets) == 1:
    raw_test = next(iter(test_datasets.values()))
else:
    raw_test = None
    print("Select DATASET_NAME to inspect raw traces.")

if raw_test is not None:
    a4_ids = a4["transaction_id"].unique()

    traces = raw_test[
        raw_test["transaction_id"].isin(a4_ids)
    ].copy()

    date_events = traces[
        traces["field_changed"]
        .astype(str)
        .str.contains("date_planned", case=False, na=False)
    ]

    columns = [
        "transaction_id",
        "sequence_no",
        "function_name",
        "action",
        "field_changed",
        "old_value",
        "new_value"
    ]

    columns = [c for c in columns if c in date_events.columns]

    display(
        date_events[columns]
        .sort_values(
            [c for c in ["transaction_id", "sequence_no"]
             if c in date_events.columns]
        )
    )


In [ ]:
features_to_check = [
    "sequence_length",
    "unique_token_count",
    "planned_date_change_count",
    "write_count",
    "repeated_functions_count",
    "consecutive_function_repeats",
    "repeated_token_count",
    "consecutive_token_repeats"
]

verification = a4.groupby("outcome")[features_to_check].agg(
    ["min", "max", "mean", "nunique"]
)

display(verification)

display(
    a4[
        ["transaction_id", "outcome"] + features_to_check
    ].sort_values(["outcome", "transaction_id"])
)